# Tutorial 2 — Categorical landscape (`Landscape` with `categorical` axes)

**Dataset:** McGuire *et al.* 2025 (JACS) — catalytic depolymerization of polyesters / polycarbonates.

Search space: **8 metal catalysts × 4 polymer substrates = 32 cells** (full factorial). Fitness `f` = `log10` rate constant (`log10_k_rate`); maximize.

Neither axis is naturally ordered — metals are chemically distinct elements; polymers are different backbones. Both should be declared `categorical`. GraphFLA's Hamming-1 neighbourhood then connects two cells iff they differ on **exactly one** axis (any value change counts as one mutation).

In [ ]:
import pandas as pd
from graphfla.landscape import Landscape
from graphfla.analysis import (
    lo_ratio, autocorrelation, fitness_distance_corr,
    classify_epistasis, all_mutation_effects,
)

df = pd.read_csv('../data/Chemistry/Depoly/depoly_rates.csv')
df[['metal', 'polymer', 'k_rate_s_inv', 'log10_k_rate']].head()

## 1. Build the landscape

Each cell has (8−1) metal-swap neighbours plus (4−1) polymer-swap neighbours = 10 neighbours. 32 cells × 10 / 2 = 160 undirected edges.

In [ ]:
X = df[['metal', 'polymer']]
f = df['log10_k_rate']

ls = Landscape(maximize=True)
ls.build_from_data(
    X, f,
    data_types={'metal': 'categorical', 'polymer': 'categorical'},
    verbose=False,
)
print(f'nodes: {ls.graph.vcount()}, edges: {ls.graph.ecount()}')

## 2. Topographical features

In [ ]:
print(f'lo_ratio        : {lo_ratio(ls):.4f}  (1/32 = single peak)')
print(f'autocorr (lag1) : {autocorrelation(ls):.4f}')
print(f'fdc (Spearman)  : {fitness_distance_corr(ls):.4f}  (negative ⇒ closer to optimum is fitter)')

print('\nEpistasis (fraction of squares):')
for k, v in classify_epistasis(ls).items():
    print(f'  {k:>26}: {v:.4f}')

## 3. Mutation-effect catalogue

`all_mutation_effects` returns one row per **(from-value, to-value)** swap, regardless of background. The columns are:

- `mutation_from`, `mutation_to` — the two values being swapped on a single axis
- `median_abs_effect` — median absolute fitness change across all backgrounds where the swap is observed
- `mean_effect` — signed mean change (positive ⇒ `to` is fitter on average)
- `p_value`, `significant` — sign test against the null of zero effect

In [ ]:
ame = all_mutation_effects(ls)
print(f'Total swap rows: {len(ame)}  (8C2 metal pairs + 4C2 polymer pairs = 28 + 6 = 34)\n')

metals = set(df['metal'].unique())
ame_metal = ame[ame['mutation_from'].isin(metals)].copy()
ame_poly  = ame[~ame['mutation_from'].isin(metals)].copy()

print('=== METAL swaps (top 5 by |mean effect|) ===')
print(ame_metal.assign(abs_mean=ame_metal['mean_effect'].abs())
              .sort_values('abs_mean', ascending=False)
              .drop(columns='abs_mean')
              .head())

print('\n=== POLYMER swaps (all 6) ===')
print(ame_poly.sort_values('mean_effect', ascending=False))

## 4. Plot fitness distribution

In [ ]:
from graphfla.plotting import draw_fitness_distribution
draw_fitness_distribution(ls)

## Notes
- Same pattern works for any all-categorical screen: drug pairs, compound × cell-line, AD × AD × AD.
- If you have **many** categorical levels (e.g. 550 compounds × 7 GPCRs in `data/Pharmacology/NpGpcr/`), the construction is identical — just larger.